# 03 - Training and Evaluation

This notebook trains candidate models, compares validation performance, and saves production artifacts.


In [ ]:
# Cell 0: Setup
from __future__ import annotations

from datetime import UTC, datetime
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from ml.pipelines.churn_train import split_by_snapshot

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent.parent

features_path = ROOT / 'ml' / 'data' / 'churn_training_features.csv'
if not features_path.exists():
    features_path = ROOT / 'ml' / 'data' / 'churn_training_dataset.csv'

selected_path = ROOT / 'ml' / 'models' / 'selected_features_notebook.json'
assert features_path.exists(), 'Run notebook 01/02 first.'
assert selected_path.exists(), 'Run notebook 02 first.'

df = pd.read_csv(features_path)
df['snapshot_date'] = pd.to_datetime(df['snapshot_date'], utc=True)
df['will_order_next_30d'] = df['will_order_next_30d'].astype(int)
selected = json.loads(selected_path.read_text(encoding='utf-8'))['selected_features']


In [ ]:
# Cell 1: Time-based split and matrices
train_df, val_df, test_df = split_by_snapshot(df)

X_train = train_df[selected]
y_train = train_df['will_order_next_30d']
X_val = val_df[selected]
y_val = val_df['will_order_next_30d']
X_test = test_df[selected]
y_test = test_df['will_order_next_30d']

print('train/val/test rows:', len(train_df), len(val_df), len(test_df))
print('selected features:', len(selected))


In [ ]:
# Cell 2: Train candidate models
candidates = {
    'logistic_regression': Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(max_iter=3000, class_weight='balanced')),
        ]
    ),
    'random_forest': Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='median')),
            (
                'clf',
                RandomForestClassifier(
                    n_estimators=500,
                    max_depth=10,
                    min_samples_leaf=8,
                    random_state=42,
                    n_jobs=-1,
                    class_weight='balanced_subsample',
                ),
            ),
        ]
    ),
}

rows = []
trained = {}
for name, model in candidates.items():
    model.fit(X_train, y_train)
    val_score = model.predict_proba(X_val)[:, 1]
    roc = roc_auc_score(y_val, val_score) if y_val.nunique() > 1 else 0.5
    pr = average_precision_score(y_val, val_score)
    rows.append({'model': name, 'val_roc_auc': roc, 'val_pr_auc': pr})
    trained[name] = model

val_table = pd.DataFrame(rows).sort_values(['val_pr_auc', 'val_roc_auc'], ascending=False)
display(val_table)


In [ ]:
# Cell 3: Select best model + threshold by F1 on validation
best_name = val_table.iloc[0]['model']
best_model = trained[best_name]

val_probs = best_model.predict_proba(X_val)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_val, val_probs)
if len(thresholds) == 0:
    threshold = 0.5
else:
    f1_vals = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    threshold = float(thresholds[int(np.nanargmax(f1_vals))])

print('Best model:', best_name)
print('Selected threshold:', threshold)


In [ ]:
# Cell 4: Test evaluation

def eval_metrics(y_true, probs, threshold):
    pred = (probs >= threshold).astype(int)
    return {
        'roc_auc': float(roc_auc_score(y_true, probs)) if y_true.nunique() > 1 else 0.5,
        'pr_auc': float(average_precision_score(y_true, probs)),
        'accuracy': float(accuracy_score(y_true, pred)),
        'precision': float(precision_score(y_true, pred, zero_division=0)),
        'recall': float(recall_score(y_true, pred, zero_division=0)),
        'f1': float(f1_score(y_true, pred, zero_division=0)),
    }

val_metrics = eval_metrics(y_val, val_probs, threshold)
test_probs = best_model.predict_proba(X_test)[:, 1]
test_metrics = eval_metrics(y_test, test_probs, threshold)

print('Validation metrics:', val_metrics)
print('Test metrics:', test_metrics)


In [ ]:
# Cell 5: Model artifacts + drift baseline
model_version = datetime.now(UTC).strftime('v%Y%m%d%H%M%S')
model_name = 'sliceiq_reorder_propensity'

artifact = {
    'model': best_model,
    'selected_features': selected,
    'threshold': threshold,
    'model_name': model_name,
    'model_version': model_version,
    'trained_at_utc': datetime.now(UTC).isoformat(),
}

model_path = ROOT / 'ml' / 'models' / 'churn_reorder_model.joblib'
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(artifact, model_path)

bins = np.linspace(0.0, 1.0, 11)
counts, _ = np.histogram(test_probs, bins=bins)
dist = (counts / max(1, counts.sum())).tolist()

drift_baseline = {
    'model_name': model_name,
    'model_version': model_version,
    'generated_at_utc': datetime.now(UTC).isoformat(),
    'bin_edges': bins.tolist(),
    'test_distribution': dist,
    'test_mean_score': float(np.mean(test_probs)),
    'test_std_score': float(np.std(test_probs)),
    'threshold': threshold,
}

baseline_path = ROOT / 'ml' / 'models' / 'churn_drift_baseline.json'
baseline_path.write_text(json.dumps(drift_baseline, indent=2), encoding='utf-8')

metrics_payload = {
    'model_name': model_name,
    'model_version': model_version,
    'best_model_family': best_name,
    'selected_features': selected,
    'threshold': threshold,
    'validation_metrics': val_metrics,
    'test_metrics': test_metrics,
    'class_balance': {
        'train_positive_rate': float(y_train.mean()),
        'val_positive_rate': float(y_val.mean()),
        'test_positive_rate': float(y_test.mean()),
    },
    'score_distribution_baseline': {
        'bin_edges': bins.tolist(),
        'test_distribution': dist,
        'test_mean_score': float(np.mean(test_probs)),
        'test_std_score': float(np.std(test_probs)),
    },
    'trained_at_utc': datetime.now(UTC).isoformat(),
}

metrics_path = ROOT / 'ml' / 'models' / 'churn_reorder_metrics.json'
metrics_path.write_text(json.dumps(metrics_payload, indent=2), encoding='utf-8')

print('Saved model:', model_path)
print('Saved metrics:', metrics_path)
print('Saved drift baseline:', baseline_path)


In [ ]:
# Cell 6: Feature importance view
clf = best_model.named_steps['clf']
if hasattr(clf, 'coef_'):
    importance = np.abs(clf.coef_[0])
elif hasattr(clf, 'feature_importances_'):
    importance = clf.feature_importances_
else:
    importance = np.zeros(len(selected))

imp_df = pd.DataFrame({'feature': selected, 'importance': importance}).sort_values('importance', ascending=False)
imp_path = ROOT / 'ml' / 'models' / 'churn_feature_importance.csv'
imp_df.to_csv(imp_path, index=False)

display(imp_df)
print('Saved feature importance:', imp_path)


## Next Notebook

Proceed to `04_causal_inference_ab_did.ipynb`.
